# EXP-NEW Multi-Population Held-Out Test Evaluation
## Evaluating Trained Checkpoint (`best.pt`) on 3 Test Populations

1. **New Held-Out Test Set**: 280 images (552 rider heads)
2. **Old Baseline Held-Out Test Set**: 829 images (1,170 rider heads)
3. **Combined Held-Out Test Set**: 1,109 images (1,722 rider heads)


In [ ]:
import subprocess
import sys
import time
import os
import json
import shutil
from pathlib import Path

print('1. INSTALLING DEPENDENCIES & VERIFYING HARDWARE')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pandas', 'matplotlib', 'seaborn'], check=True)

import torch
import ultralytics
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}, Ultralytics: {ultralytics.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise SystemError('CUDA not available!')


In [ ]:
print('2. LOCATING DATASETS & MODEL WEIGHTS')
input_dir = Path('/kaggle/input')
new_ds_path = None
old_ds_path = None
weights_path = None

for p in input_dir.rglob('*'):
    if p.is_dir() and 'helmet-dataset-expansion-v2' in p.name.lower():
        new_ds_path = p
    elif p.is_dir() and 'old-dataset-v1-fixed' in p.name.lower():
        old_ds_path = p
    elif p.is_file() and p.name == 'best.pt':
        weights_path = p

if not new_ds_path:
    for p in input_dir.glob('**/*'):
        if (p / 'images/test').exists():
            new_ds_path = p
            break

if not old_ds_path:
    for p in input_dir.glob('**/*'):
        if (p / 'test/images').exists():
            old_ds_path = p
            break

print(f'New Dataset Root: {new_ds_path}')
print(f'Old Dataset Root: {old_ds_path}')
print(f'Weights Path:     {weights_path}')

assert new_ds_path and new_ds_path.exists(), 'New dataset not found!'
assert old_ds_path and old_ds_path.exists(), 'Old dataset not found!'
assert weights_path and weights_path.exists(), 'Weights not found!'

n_test = len(list((new_ds_path / 'images/test').glob('*.jpg')))
o_test = len(list((old_ds_path / 'test/images').glob('*.jpg')))
print(f'New Test Images: {n_test} (expected 280)')
print(f'Old Test Images: {o_test} (expected 829)')
assert n_test == 280, f'Expected 280, got {n_test}'
assert o_test == 829, f'Expected 829, got {o_test}'


In [ ]:
print('3. RUNNING MULTI-POPULATION EVALUATIONS')
work_dir = Path('/kaggle/working')
eval_model = YOLO(str(weights_path))

def extract_metrics(metrics, name):
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = float(2 * p * r / (p + r + 1e-16))
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    per_class = {}
    names = metrics.names
    for i, cls_id in enumerate(metrics.box.ap_class_index):
        cls_name = names.get(cls_id, f'class_{cls_id}')
        cls_p = float(metrics.box.p[i]) if len(metrics.box.p) > i else 0.0
        cls_r = float(metrics.box.r[i]) if len(metrics.box.r) > i else 0.0
        cls_f1 = float(2 * cls_p * cls_r / (cls_p + cls_r + 1e-16))
        cls_map50 = float(metrics.box.ap50[i]) if len(metrics.box.ap50) > i else 0.0
        cls_map50_95 = float(metrics.box.ap[i]) if len(metrics.box.ap) > i else 0.0
        per_class[cls_name] = {
            'class_id': int(cls_id),
            'precision': round(cls_p, 4),
            'recall': round(cls_r, 4),
            'f1': round(cls_f1, 4),
            'map50': round(cls_map50, 4),
            'map50_95': round(cls_map50_95, 4)
        }
    cm = None
    if hasattr(metrics, 'confusion_matrix') and hasattr(metrics.confusion_matrix, 'matrix'):
        cm = metrics.confusion_matrix.matrix.tolist()
    return {
        'evaluation_name': name,
        'overall': {
            'precision': round(p, 4),
            'recall': round(r, 4),
            'f1': round(f1, 4),
            'map50': round(map50, 4),
            'map50_95': round(map50_95, 4)
        },
        'per_class': per_class,
        'confusion_matrix': cm,
        'speed_ms': {k: round(v, 2) for k, v in metrics.speed.items()}
    }

# --- EVALUATION 1: NEW HELD-OUT TEST SET (280 images) ---
print('>>> Evaluating on NEW Test Set (280 images)...')
eval_new_yaml = work_dir / 'eval_new.yaml'
with open(eval_new_yaml, 'w') as f:
    f.write(f'path: {new_ds_path}\ntrain: images/test\nval: images/test\nnames:\n  0: no_helmet\n  1: helmet\nnc: 2\n')
res_new = eval_model.val(data=str(eval_new_yaml), split='val', batch=32, imgsz=640, device=0, plots=True, project='/kaggle/working/eval_runs', name='new_test')
metrics_new = extract_metrics(res_new, 'New Held-Out Test Set (280 images)')

# --- EVALUATION 2: OLD BASELINE HELD-OUT TEST SET (829 images) ---
print('>>> Evaluating on OLD Test Set (829 images)...')
eval_old_yaml = work_dir / 'eval_old.yaml'
with open(eval_old_yaml, 'w') as f:
    f.write(f'path: {old_ds_path}\ntrain: test/images\nval: test/images\nnames:\n  0: no_helmet\n  1: helmet\nnc: 2\n')
res_old = eval_model.val(data=str(eval_old_yaml), split='val', batch=32, imgsz=640, device=0, plots=True, project='/kaggle/working/eval_runs', name='old_test')
metrics_old = extract_metrics(res_old, 'Old Held-Out Test Set (829 images)')

# --- EVALUATION 3: COMBINED HELD-OUT TEST SET (1,109 images) ---
print('>>> Materializing & Evaluating on COMBINED Test Set (1,109 images)...')
comb_dir = work_dir / 'combined_test_dataset'
comb_imgs = comb_dir / 'images/val'
comb_lbls = comb_dir / 'labels/val'
comb_imgs.mkdir(parents=True, exist_ok=True)
comb_lbls.mkdir(parents=True, exist_ok=True)

for img_p in (new_ds_path / 'images/test').glob('*.jpg'):
    dst_i = comb_imgs / f'new_{img_p.name}'
    lbl_p = new_ds_path / 'labels/test' / f'{img_p.stem}.txt'
    dst_l = comb_lbls / f'new_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

for img_p in (old_ds_path / 'test/images').glob('*.jpg'):
    dst_i = comb_imgs / f'old_{img_p.name}'
    lbl_p = old_ds_path / 'test/labels' / f'{img_p.stem}.txt'
    dst_l = comb_lbls / f'old_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

eval_comb_yaml = work_dir / 'eval_comb.yaml'
with open(eval_comb_yaml, 'w') as f:
    f.write(f'path: {comb_dir}\ntrain: images/val\nval: images/val\nnames:\n  0: no_helmet\n  1: helmet\nnc: 2\n')
res_comb = eval_model.val(data=str(eval_comb_yaml), split='val', batch=32, imgsz=640, device=0, plots=True, project='/kaggle/working/eval_runs', name='comb_test')
metrics_comb = extract_metrics(res_comb, 'Combined Held-Out Test Set (1,109 images)')


In [ ]:
print('4. SAVING RESULTS JSON & REPORTING TABLES')
summary = {
    'experiment': 'EXP-NEW',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'checkpoint': str(weights_path),
    'evaluations': {
        'new_held_out_test': metrics_new,
        'old_held_out_test': metrics_old,
        'combined_held_out_test': metrics_comb
    }
}
summary_file = work_dir / 'exp_new_evaluation_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved summary to {summary_file}')

print('=' * 85)
print(f'{"Test Population":<30} | {"Precision":<10} | {"Recall":<10} | {"F1":<10} | {"mAP50":<10} | {"mAP50-95":<10}')
print('-' * 85)
for k, name in [('new_held_out_test', 'New Test (280)'), ('old_held_out_test', 'Old Test (829)'), ('combined_held_out_test', 'Combined Test (1,109)')]:
    m = summary['evaluations'][k]['overall']
    print(f'{name:<30} | {m["precision"]:10.4f} | {m["recall"]:10.4f} | {m["f1"]:10.4f} | {m["map50"]:10.4f} | {m["map50_95"]:10.4f}')
print('=' * 85)
